# 04 - Custom Evaluators

Built-ins should be the default. Create a custom evaluator when your failure mode is specific to your product, domain, schema, or policy.

You will:

1. Define one failure mode per evaluator.
2. Configure a binary LLM-as-a-Judge evaluator.
3. Unit-test a deterministic code-based evaluator locally.
4. Calibrate a judge against human-labeled examples.

**Estimated time:** 45-60 minutes  
**Creates AWS resources:** Optional custom evaluator and Lambda resources.

## 1. Evaluator design rules

Prefer:

- one failure mode
- binary pass/fail when the operational decision is binary
- explicit definitions and edge cases
- structured outputs
- a human-labeled calibration set

Avoid one broad prompt that scores "helpfulness, accuracy, clarity, professionalism, and completeness" on a 1-5 scale. It is hard to determine why the score changed or whether the judge is reliable.

## 2. Focused LLM-as-a-Judge

Failure mode: **factual consistency with supplied reference facts**.

The evaluator should pass only when city numbers and comparisons agree with the reference context. Style, verbosity, and tone are out of scope.

In [ ]:
FACTUAL_CONSISTENCY_INSTRUCTIONS = '''
Evaluate only factual consistency with the supplied reference information.

PASS when every city fact, number, and comparison in the assistant response
agrees with the reference information in {context}. Reasonable rounding is
allowed when the user asks for an approximation.

FAIL when the response gives a wrong number, reverses a comparison, substitutes
a different city or state, invents data for a missing city, or does not answer
the factual question.

Ignore writing style, tone, and verbosity. Return the configured rating label
and a concise explanation tied to this criterion only.
'''.strip()

print(FACTUAL_CONSISTENCY_INSTRUCTIONS)

Add the evaluator to the local project configuration. The model ID below is an example; verify a supported evaluator model or inference profile in your region.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

MODULE_ROOT = Path.cwd()
JUDGE_MODEL = os.getenv(
    "AGENTCORE_JUDGE_MODEL_ID",
    "global.anthropic.claude-sonnet-4-6",
)
CREATE_LLM_EVALUATOR = False

config_path = MODULE_ROOT / "agentcore" / "agentcore.json"
project_config = json.loads(config_path.read_text())
existing_names = {
    item["name"] for item in project_config.get("evaluators", [])
}

if CREATE_LLM_EVALUATOR and "FactualConsistency" not in existing_names:
    subprocess.run(
        [
            "agentcore",
            "add",
            "evaluator",
            "--name",
            "FactualConsistency",
            "--level",
            "TRACE",
            "--model",
            JUDGE_MODEL,
            "--instructions",
            FACTUAL_CONSISTENCY_INSTRUCTIONS,
            "--rating-scale",
            "pass-fail",
        ],
        cwd=MODULE_ROOT,
        check=True,
    )
    print("Evaluator added locally. Run agentcore deploy to create it.")
else:
    print(
        "Set CREATE_LLM_EVALUATOR=True to add the evaluator, "
        "or keep reading to unit-test a code evaluator locally."
    )

Adding an evaluator changes `agentcore.json`; deploying creates the AWS resource. Defining and reviewing evaluator instructions does not require a Runtime deployment.

After deployment:

```bash
agentcore run eval \
  --runtime CityAnalyst \
  --evaluator FactualConsistency \
  --session-id <session-id> \
  --expected-response "Seattle population is 780995."
```

## 3. Deterministic code-based evaluator

Failure mode: **the final response violates the required JSON schema**.

This should be code, not an LLM judge. A deterministic check is cheaper, repeatable, and easier to debug.

In [ ]:
from bedrock_agentcore.evaluation.custom_code_based_evaluators import EvaluatorInput
from evaluators.response_format_evaluator import (
    handler,
    validate_response_schema,
)

valid_response = json.dumps(
    {
        "answer": "Seattle has 780995 residents.",
        "cities": [
            {
                "city": "Seattle",
                "state": "WA",
                "population": 780995,
                "land_area_mi2": 83.8,
                "density_per_mi2": 9319.7,
            }
        ],
        "tools_used": ["lookup_city"],
    }
)
invalid_response = "Seattle has 780995 residents."

print(validate_response_schema(valid_response))
print(validate_response_schema(invalid_response))

The decorator exposes the original typed function as `handler.unwrapped`, which makes local tests independent of Lambda event parsing.

In [ ]:
fixture_spans = json.loads(Path("data/sample_spans.json").read_text())
evaluator_input = EvaluatorInput(
    evaluation_level="TRACE",
    session_spans=fixture_spans,
    target_trace_id=fixture_spans[0]["traceId"],
    evaluator_id="local-response-format",
    evaluator_name="ResponseFormat",
)

local_result = handler.unwrapped(evaluator_input, context=None)
local_result.model_dump()

In [ ]:
failing_spans = json.loads(Path("data/sample_spans.json").read_text())
failing_spans[-1]["attributes"]["gen_ai.response.content"] = (
    "Seattle has 780995 residents."
)
failing_input = EvaluatorInput(
    evaluation_level="TRACE",
    session_spans=failing_spans,
    target_trace_id=failing_spans[0]["traceId"],
    evaluator_id="local-response-format",
    evaluator_name="ResponseFormat",
)

failing_result = handler.unwrapped(failing_input, context=None)
failing_result.model_dump()

## 4. Register the code evaluator

Two production paths are common:

1. Let the interactive CLI scaffold and manage a Python Lambda evaluator:

   ```bash
   agentcore add evaluator
   ```

   Choose `code-based`, then point the managed evaluator at `evaluators/response_format_evaluator.py`.

2. Register an existing Lambda non-interactively:

   ```bash
   agentcore add evaluator \
     --name ResponseFormat \
     --level TRACE \
     --type code-based \
     --lambda-arn <LAMBDA_ARN> \
     --timeout 30
   ```

Deploy only after the local fixtures pass. The deployment step should wire known-good logic into AgentCore, not become the evaluator debugging loop.

## 5. Validate the judge, not just the agent

A judge can be confidently wrong. Use a separate human-labeled benchmark to measure it.

The checked-in file contains clear pass/fail examples for factual consistency.

In [ ]:
validation_examples = [
    json.loads(line)
    for line in Path("data/judge_validation.jsonl").read_text().splitlines()
    if line.strip()
]
pd_rows = [
    {
        "id": item["id"],
        "human_label": item["human_label"],
        "question": item["question"],
        "response": item["response"],
    }
    for item in validation_examples
]

import pandas as pd
pd.DataFrame(pd_rows)

In [ ]:
def judge_scorecard(records):
    evaluated = [
        item for item in records
        if item.get("judge_label") in {"pass", "fail"}
    ]
    if not evaluated:
        raise ValueError("Add judge_label values before computing the scorecard.")

    true_pass = [item for item in evaluated if item["human_label"] == "pass"]
    true_fail = [item for item in evaluated if item["human_label"] == "fail"]
    accuracy = sum(
        item["human_label"] == item["judge_label"]
        for item in evaluated
    ) / len(evaluated)
    tpr = sum(item["judge_label"] == "pass" for item in true_pass) / len(true_pass)
    tnr = sum(item["judge_label"] == "fail" for item in true_fail) / len(true_fail)
    return {
        "examples": len(evaluated),
        "accuracy": accuracy,
        "true_positive_rate": tpr,
        "true_negative_rate": tnr,
    }

print(
    "Run the deployed custom evaluator on each fixture, store judge_label, "
    "then call judge_scorecard(validation_examples)."
)

Calibration workflow:

1. reserve examples for few-shot guidance, development, and held-out testing
2. prevent the same question family from leaking across splits
3. inspect every judge-human disagreement
4. revise definitions and examples, not the labels
5. test repeatability across multiple judge runs
6. record the final true-positive and true-negative rates

The foundational notebook [Evaluating Your Judge](../../Foundational%20Evaluations/02-quality-metrics/03_Evaluating_your_Judge.ipynb) covers this process in depth.

## 6. Checkpoint

You should now be able to decide:

- built-in or custom
- LLM judge or deterministic code
- session, trace, or tool-call level
- what labeled data is needed to trust the evaluator

Continue to [05 - Batch and Online Evaluation](05-batch-and-online-evaluation.ipynb) to use these evaluators in regression gates and sampled live monitoring.